**Crypto Coin Recommender System ด้วย Graph (Neo4j)**

<br>แนวคิดคือ ผู้ใช้ถือหรือสนใจเหรียญอะไร → หา User คนอื่นที่พอร์ตคล้ายกัน → ดูว่าคนเหล่านั้นถือเหรียญอะไรอีก → แนะนำเหรียญนั้นกลับมา

เวอร์ชันนี้เก็บข้อมูลไว้ใน **Neo4j Aura (online)** และเดินกราฟด้วย **Cypher** แทน NetworkX

> หมายเหตุ: โน้ตบุ๊กนี้เป็นตัวอย่างเชิงวิชาการเรื่องโครงสร้าง Graph เท่านั้น ไม่ใช่คำแนะนำการลงทุน

ตัวอย่างสถานการณ์ มีผู้ใช้ 10 คน <br>

```
Anan   Fah
Bee    Gun
Chai   Ice
Dao    Jane
Earn   Kong
```

และมีเหรียญ Crypto 8 เหรียญ

```
BTC   ADA
ETH   XRP
BNB   DOGE
SOL   PEPE
```

สมมติข้อมูลความสนใจ (ถือ / Watchlist) รวม 26 เส้นเชื่อม

```
Anan → BTC, ETH
Bee  → BTC, ETH, SOL
Chai → BTC, BNB
Dao  → SOL, DOGE
Earn → ETH, SOL, ADA
Fah  → BTC, XRP
Gun  → DOGE, PEPE
Ice  → ETH, ADA, XRP
Jane → BTC, ETH, BNB, SOL
Kong → SOL, PEPE, DOGE
```

จะเห็นว่าเริ่มมีกลุ่มก้อนชัดขึ้น เช่น กลุ่มสาย Layer1 (Anan, Bee, Earn, Jane)
และกลุ่มสาย Meme (Dao, Gun, Kong)

```
เราจะสังเกตได้ว่า
Anan ถือ BTC และ ETH
คนที่ถือเหรียญซ้ำกับ Anan มีหลายคน
Bee, Chai, Earn, Fah, Ice, Jane
และคนกลุ่มนี้ยังถือ SOL กันหลายคน
(Bee, Earn, Jane)
แต่ Anan ยังไม่มี SOL
ระบบจึงแนะนำ SOL ให้ Anan เป็นอันดับหนึ่ง
ยิ่งผู้ใช้เยอะ คะแนนยิ่งแยกอันดับได้ชัดขึ้น
เพราะเหรียญเดียวกันถูกเดินไปถึงหลายเส้นทาง
```

การออกแบบ Graph

```
เราแบ่ง Node เป็น 2 ประเภท
  (:User {name})
  (:Coin {symbol})

Relationship คือ
  HOLDS      ผู้ใช้ถือเหรียญ
  FRIEND_OF  ผู้ใช้ 2 คนถือเหรียญเดียวกันอย่างน้อย 1 เหรียญ (สร้างจาก HOLDS)

ดังนั้น
  (User)-[:HOLDS]->(Coin)
  (User)-[:FRIEND_OF {shared_coins, weight}]->(User)
เช่น
  (:User {name:'Anan'})-[:HOLDS]->(:Coin {symbol:'BTC'})
  (:User {name:'Anan'})-[:FRIEND_OF {shared_coins:['BTC','ETH'], weight:2}]->(:User {name:'Bee'})
```

ใน Neo4j ความสัมพันธ์มีทิศทางเสมอ แต่ตอน Query เราเดินย้อนทิศได้ด้วย `<-[:HOLDS]-`
หรือไม่สนทิศเลยด้วย `-[:FRIEND_OF]-` จึงไม่ต้องสร้างเส้นกลับเหมือน Undirected Graph

**1. ติดตั้งและเชื่อมต่อ Neo4j online server**

In [ ]:
!pip install -q neo4j

In [ ]:
import neo4j

print("Neo4j Python Driver Version:", neo4j.__version__)

In [ ]:
from getpass import getpass
from neo4j import GraphDatabase

URI = "neo4j+s://27edd787.databases.neo4j.io"
USERNAME = "27edd787"
PASSWORD = getpass("Password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected successfully")

หา home database แล้วเก็บชื่อไว้ใช้กับทุก Query

In [ ]:
result = driver.execute_query(
    "SHOW HOME DATABASE"
)

DATABASE = result.records[0]["name"]

print("Database =", DATABASE)

**2. ล้างข้อมูล Crypto เดิม (ถ้ามี)**

ลบเฉพาะ Node ของโปรเจกต์นี้ (User / Coin / Category) เพื่อให้รันซ้ำได้โดยไม่กระทบข้อมูลอื่นในฐานข้อมูล

In [ ]:
driver.execute_query(
    """
    MATCH (n)

    WHERE n:User OR n:Coin OR n:Category

    DETACH DELETE n
    """,
    database_=DATABASE
)

print("ล้างข้อมูลเดิมเรียบร้อย")

**3. สร้าง Constraint** (กันข้อมูลซ้ำ และทำให้ MATCH เร็วขึ้น)

In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT user_name_unique
    IF NOT EXISTS

    FOR (u:User)

    REQUIRE u.name IS UNIQUE
    """,
    database_=DATABASE
)

driver.execute_query(
    """
    CREATE CONSTRAINT coin_symbol_unique
    IF NOT EXISTS

    FOR (c:Coin)

    REQUIRE c.symbol IS UNIQUE
    """,
    database_=DATABASE
)

print("User / Coin constraint created")

**4. สร้าง Node ชนิดที่ 1: ผู้ใช้ (10 คน)**

In [ ]:
users = [
    "Anan",
    "Bee",
    "Chai",
    "Dao",
    "Earn",
    "Fah",
    "Gun",
    "Ice",
    "Jane",
    "Kong"
]

In [ ]:
# UNWIND = วนลูปรายการใน Cypher / MERGE = สร้างถ้ายังไม่มี (รันซ้ำไม่เกิดข้อมูลซ้ำ)

result = driver.execute_query(
    """
    UNWIND $users AS name

    MERGE (u:User {name: name})

    RETURN count(u) AS total
    """,

    users=users,
    database_=DATABASE
)

print("จำนวน User ที่ประมวลผล =", result.records[0]["total"])

**5. สร้าง Node ชนิดที่ 2: เหรียญ Crypto (8 เหรียญ)**

In [ ]:
coins = [
    "BTC",
    "ETH",
    "BNB",
    "SOL",
    "ADA",
    "XRP",
    "DOGE",
    "PEPE"
]

In [ ]:
result = driver.execute_query(
    """
    UNWIND $coins AS symbol

    MERGE (c:Coin {symbol: symbol})

    RETURN count(c) AS total
    """,

    coins=coins,
    database_=DATABASE
)

print("จำนวน Coin ที่ประมวลผล =", result.records[0]["total"])

**6. สร้าง Relationship HOLDS (ผู้ใช้ถือเหรียญนี้)**

In [ ]:
holds = [
    ("Anan", "BTC"),
    ("Anan", "ETH"),

    ("Bee", "BTC"),
    ("Bee", "ETH"),
    ("Bee", "SOL"),

    ("Chai", "BTC"),
    ("Chai", "BNB"),

    ("Dao", "SOL"),
    ("Dao", "DOGE"),

    ("Earn", "ETH"),
    ("Earn", "SOL"),
    ("Earn", "ADA"),

    ("Fah", "BTC"),
    ("Fah", "XRP"),

    ("Gun", "DOGE"),
    ("Gun", "PEPE"),

    ("Ice", "ETH"),
    ("Ice", "ADA"),
    ("Ice", "XRP"),

    ("Jane", "BTC"),
    ("Jane", "ETH"),
    ("Jane", "BNB"),
    ("Jane", "SOL"),

    ("Kong", "SOL"),
    ("Kong", "PEPE"),
    ("Kong", "DOGE")
]

In [ ]:
# Neo4j driver รับ list ของ dict ได้สะดวกกว่า tuple จึงแปลงก่อนส่ง
holds_data = [
    {"user": u, "coin": c}
    for u, c in holds
]

driver.execute_query(
    """
    UNWIND $holds AS row

    MATCH
        (u:User {name: row.user}),
        (c:Coin {symbol: row.coin})

    MERGE
        (u)-[:HOLDS]->(c)
    """,

    holds=holds_data,
    database_=DATABASE
)

print("สร้าง HOLDS สำเร็จ")

ตรวจจำนวน Node และ Relationship (ควรได้ 18 Node, 26 Relationship)

In [ ]:
result = driver.execute_query(
    """
    MATCH (n)

    WHERE n:User OR n:Coin

    OPTIONAL MATCH (n)-[r:HOLDS]->()

    RETURN
        count(DISTINCT n) AS nodes,
        count(r) AS relationships
    """,
    database_=DATABASE
)

print(result.records[0].data())

**7. วาด Graph (ดึงข้อมูลจาก Neo4j มาวาด)**

ดึงเส้น HOLDS ทั้งหมดจากฐานข้อมูล แล้วใช้ NetworkX + matplotlib วาดเพื่อดูด้วยตา (ใช้แค่วาด ไม่ได้ใช้คำนวณ)

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

result = driver.execute_query(
    """
    MATCH (u:User)-[:HOLDS]->(c:Coin)

    RETURN
        u.name AS user,
        c.symbol AS coin
    """,
    database_=DATABASE
)

edges_df = pd.DataFrame(
    [record.data() for record in result.records]
)

G = nx.Graph()
G.add_nodes_from(edges_df["user"].unique(), node_type="user")
G.add_nodes_from(edges_df["coin"].unique(), node_type="coin")
G.add_edges_from(edges_df[["user", "coin"]].itertuples(index=False))

plt.figure(figsize=(13, 9))

pos = nx.spring_layout(G, k=0.8, seed=42)

# ฟ้า = ผู้ใช้ / ส้ม = เหรียญ
colors = [
    "#87CEFA" if G.nodes[n]["node_type"] == "user" else "#FFD27F"
    for n in G.nodes
]

nx.draw(
    G,
    pos,
    with_labels=True,
    node_color=colors,
    node_size=2200,
    font_size=9
)

plt.show()

หรือดูใน Neo4j Aura → Query แล้วรัน

```
MATCH p = (:User)-[:HOLDS]->(:Coin)
RETURN p
```

**8. สร้าง Relationship FRIEND_OF (ใครถือเหรียญเดียวกันบ้าง)**

แทนที่จะพิมพ์รายการเพื่อนเอง เราให้ Neo4j หาจาก `HOLDS` ที่มีอยู่แล้ว:

```
(a:User)-[:HOLDS]->(c:Coin)<-[:HOLDS]-(b:User)
```

- `a.name < b.name` = สร้างเส้นเดียวต่อคู่ (กัน Anan→Bee กับ Bee→Anan ซ้ำกัน)
- `collect(c.symbol)` = เก็บเหรียญที่ถือร่วมกันไว้ใน property `shared_coins`
- `weight` = จำนวนเหรียญที่ถือร่วมกัน ยิ่งมากยิ่งพอร์ตคล้ายกัน

In [ ]:
driver.execute_query(
    """
    MATCH
        (a:User)
        -[:HOLDS]->
        (c:Coin)
        <-[:HOLDS]-
        (b:User)

    WHERE a.name < b.name

    WITH
        a,
        b,
        collect(c.symbol) AS shared

    MERGE
        (a)-[f:FRIEND_OF]->(b)

    SET
        f.shared_coins = shared,
        f.weight = size(shared)
    """,
    database_=DATABASE
)

result = driver.execute_query(
    """
    MATCH ()-[f:FRIEND_OF]->()
    RETURN count(f) AS total
    """,
    database_=DATABASE
)

print("สร้าง FRIEND_OF สำเร็จ =", result.records[0]["total"], "คู่")

ดู FRIEND_OF ทั้งหมดเป็นตาราง (ควรได้ 27 คู่)

In [ ]:
result = driver.execute_query(
    """
    MATCH (a:User)-[f:FRIEND_OF]->(b:User)

    RETURN
        a.name AS user1,
        b.name AS user2,
        f.shared_coins AS shared_coins,
        f.weight AS weight

    ORDER BY
        weight DESC,
        user1,
        user2
    """,
    database_=DATABASE
)

pd.DataFrame(
    [record.data() for record in result.records]
)

คำถาม: Anan เป็นเพื่อน (ถือเหรียญเดียวกัน) กับใครบ้าง?

ใช้ `-[:FRIEND_OF]-` แบบไม่มีลูกศร เพื่อเดินได้ทั้งสองทิศ

In [ ]:
result = driver.execute_query(
    """
    MATCH (me:User {name: $name})-[f:FRIEND_OF]-(friend:User)

    RETURN
        friend.name AS friend,
        f.shared_coins AS shared_coins,
        f.weight AS weight

    ORDER BY
        weight DESC,
        friend
    """,

    name="Anan",
    database_=DATABASE
)

pd.DataFrame(
    [record.data() for record in result.records]
)

ผลที่ควรได้: Bee และ Jane ถือ BTC, ETH ร่วมกับ Anan (weight 2)
ส่วน Chai, Fah ร่วมแค่ BTC และ Earn, Ice ร่วมแค่ ETH (weight 1)

วาด Graph เฉพาะ User ↔ User (ความหนาของเส้น = weight)

In [ ]:
result = driver.execute_query(
    """
    MATCH (a:User)-[f:FRIEND_OF]->(b:User)

    RETURN
        a.name AS a,
        b.name AS b,
        f.weight AS weight
    """,
    database_=DATABASE
)

F = nx.Graph()
F.add_nodes_from(users)

for record in result.records:
    F.add_edge(record["a"], record["b"], weight=record["weight"])

plt.figure(figsize=(10, 8))

pos_f = nx.spring_layout(F, k=0.9, seed=42)

nx.draw(
    F,
    pos_f,
    with_labels=True,
    node_color="#87CEFA",
    node_size=2000,
    font_size=9,
    width=[F[u][v]["weight"] * 1.5 for u, v in F.edges]
)

nx.draw_networkx_edge_labels(
    F,
    pos_f,
    edge_labels=nx.get_edge_attributes(F, "weight"),
    font_size=8
)

plt.show()

หรือดูใน Neo4j Aura → Query แล้วรัน

```
MATCH p = (:User)-[:FRIEND_OF]-(:User)
RETURN p
```

คำถามแรก <br>

1. ถามว่า Anan ถือเหรียญอะไรบ้าง? (เดิน 1 hop)

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User {name: $name})-[:HOLDS]->(c:Coin)

    RETURN c.symbol AS coin

    ORDER BY coin
    """,

    name="Anan",
    database_=DATABASE
)

anan_coins = [record["coin"] for record in result.records]

print(anan_coins)

ตรงนี้อธิบายได้ว่า Neighbor ของ Anan คือ Node ที่เชื่อมกับ Anan ผ่าน `HOLDS`

คำถามที่ 2. หา User ที่ถือเหรียญเหมือน Anan <br>

Anan ถือ BTC ETH <br>

เดิน 2 hop: `Anan -[:HOLDS]-> Coin <-[:HOLDS]- User อื่น`

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (me:User {name: $name})
        -[:HOLDS]->
        (c:Coin)
        <-[:HOLDS]-
        (other:User)

    WHERE other <> me

    RETURN
        c.symbol AS coin,
        collect(other.name) AS similar_users

    ORDER BY coin
    """,

    name="Anan",
    database_=DATABASE
)

for record in result.records:
    print("Coin:", record["coin"])
    for user in record["similar_users"]:
        print("  Similar user:", user)

**สร้าง Recommendation แบบง่าย**

```
แนวคิดคือ

Anan
 ↓  HOLDS
เหรียญที่ Anan ถือ
 ↓  <-HOLDS
User ที่ถือเหรียญเดียวกัน
 ↓  HOLDS
เหรียญอื่นที่ User เหล่านั้นถือ
 ↓
แนะนำให้ Anan
```

ใน Cypher เขียนทั้ง 3 hop ได้ใน `MATCH` เดียว
- `WHERE NOT EXISTS { (me)-[:HOLDS]->(rec) }` = ตัดเหรียญที่ Anan มีแล้วออก
- `count(*)` = นับจำนวนเส้นทางที่เดินไปถึงเหรียญนั้น (เทียบเท่า `Counter` ในเวอร์ชัน NetworkX)

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (me:User {name: $name})
        -[:HOLDS]->
        (c:Coin)
        <-[:HOLDS]-
        (other:User)
        -[:HOLDS]->
        (rec:Coin)

    WHERE other <> me
      AND NOT EXISTS {
          MATCH (me)-[:HOLDS]->(rec)
      }

    RETURN
        rec.symbol AS coin,
        count(*) AS score

    ORDER BY
        score DESC,
        coin
    """,

    name="Anan",
    database_=DATABASE
)

pd.DataFrame(
    [record.data() for record in result.records]
)

ผลที่ได้คือ `SOL 5, BNB 3, ADA 2, XRP 2`

SOL ได้ 5 คะแนน เพราะเดินจาก Anan ไปถึง SOL ได้ถึง 5 เส้นทาง
(ผ่าน BTC → Bee/Jane และ ETH → Bee/Earn/Jane)

DOGE และ PEPE ไม่ถูกแนะนำ เพราะไม่มีใครในกลุ่มที่พอร์ตทับซ้อนกับ Anan ถือเหรียญสาย Meme เลย

ยิ่งคะแนนมาก แปลว่าเชื่อมโยงกันหลายเส้นทาง ไม่ใช่บังเอิญตรงกันแค่จุดเดียว

**นำไปสร้างเป็นฟังก์ชัน**

In [ ]:
def recommend_coins(target_user, top_n=None):
    """
    แนะนำเหรียญให้ target_user ด้วยการเดินกราฟ 3 hop ใน Neo4j

        target_user -> เหรียญที่ถือ -> คนอื่นที่ถือเหมือนกัน -> เหรียญใหม่

    คืนค่า: list ของ (เหรียญ, คะแนน) เรียงจากมากไปน้อย
    top_n: ถ้ากำหนด จะตัดเฉพาะอันดับต้น ๆ ตั้งแต่ใน Cypher (LIMIT)
    """

    query = """
    MATCH
        (me:User {name: $name})
        -[:HOLDS]->
        (:Coin)
        <-[:HOLDS]-
        (other:User)
        -[:HOLDS]->
        (rec:Coin)

    WHERE other <> me
      AND NOT EXISTS {
          MATCH (me)-[:HOLDS]->(rec)
      }

    RETURN
        rec.symbol AS coin,
        count(*) AS score

    ORDER BY
        score DESC,
        coin
    """

    if top_n is not None:
        query += "\nLIMIT $top_n"

    result = driver.execute_query(
        query,
        name=target_user,
        top_n=top_n,
        database_=DATABASE
    )

    return [
        (record["coin"], record["score"])
        for record in result.records
    ]

In [ ]:
# เรียกใช้งานทีละคน
# ผลที่ได้: SOL 5 คะแนน เพราะเดินจาก Anan ไปถึง SOL ได้ถึง 5 เส้นทาง
#   ผ่าน BTC -> Bee, Jane
#   ผ่าน ETH -> Bee, Earn, Jane

result = recommend_coins("Anan")
print(result)

In [ ]:
# ดูผลของผู้ใช้ทุกคน
# สังเกต Gun ที่ถือแต่ DOGE กับ PEPE จะได้ผลแนะนำน้อยมาก
# เพราะเชื่อมกับส่วนอื่นของกราฟผ่าน Kong ทางเดียว (ปัญหา Cold Start)

for user in users:
    print(user, "->", recommend_coins(user))

In [ ]:
# ในระบบจริงจะไม่แสดงทั้งหมด แต่ตัดเอาเฉพาะอันดับต้น ๆ (ใช้ LIMIT ใน Cypher)
TOP_N = 3

for user in users:
    top = recommend_coins(user, top_n=TOP_N)
    print(f"{user:5s} -> {top}")

**แนะนำเหรียญผ่าน FRIEND_OF**

เมื่อมี FRIEND_OF แล้ว เดินแค่ 2 hop ก็แนะนำได้ (เหมือน Step 18 ใน PraceticeNeo4jDBOnline)

```
Anan -[:FRIEND_OF]- เพื่อน -[:HOLDS]-> เหรียญที่ Anan ยังไม่มี
```

มีคะแนน 2 แบบ
- `friend_score` = จำนวนเพื่อนที่ถือเหรียญนั้น
- `weighted_score` = รวม `weight` ของเพื่อนเหล่านั้น ซึ่งจะได้ค่า **เท่ากับคะแนนแบบ 3 hop เดิม** เพราะ weight คือจำนวนเหรียญร่วม = จำนวนเส้นทางที่เดินผ่าน

In [ ]:
def recommend_by_friends(target_user, top_n=None):
    """
    แนะนำเหรียญจากสิ่งที่เพื่อน (ผู้ใช้ที่ถือเหรียญเดียวกัน) ถือ
    แต่ target_user ยังไม่มี

        target_user -[:FRIEND_OF]- friend -[:HOLDS]-> เหรียญใหม่
    """

    query = """
    MATCH
        (me:User {name: $name})
        -[f:FRIEND_OF]-
        (friend:User)
        -[:HOLDS]->
        (rec:Coin)

    WHERE NOT EXISTS {
        MATCH (me)-[:HOLDS]->(rec)
    }

    RETURN
        rec.symbol AS coin,
        count(DISTINCT friend) AS friend_score,
        sum(f.weight) AS weighted_score,
        collect(friend.name) AS from_friends

    ORDER BY
        weighted_score DESC,
        friend_score DESC,
        coin
    """

    if top_n is not None:
        query += "\nLIMIT $top_n"

    result = driver.execute_query(
        query,
        name=target_user,
        top_n=top_n,
        database_=DATABASE
    )

    return pd.DataFrame(
        [record.data() for record in result.records]
    )

In [ ]:
# ผลที่ควรได้ของ Anan
#   SOL : เพื่อน 3 คน (Bee, Earn, Jane) weighted 5  <- ตรงกับ 3 hop เดิม
#   BNB : เพื่อน 2 คน (Chai, Jane)      weighted 3
#   ADA : เพื่อน 2 คน (Earn, Ice)       weighted 2
#   XRP : เพื่อน 2 คน (Fah, Ice)        weighted 2

recommend_by_friends("Anan")

In [ ]:
# Gun มีเพื่อนแค่ Dao กับ Kong จึงได้คำแนะนำแค่ SOL
recommend_by_friends("Gun")

**ส่วนขยาย (ถ้าอยากได้มากกว่า Movie version)**

เพิ่ม Node ประเภทที่ 3 คือ `Category` เช่น Layer1 / Exchange / Meme

แล้วเชื่อม `(Coin)-[:IN_CATEGORY]->(Category)`

ทำให้แนะนำได้แม้ User ไม่มีเพื่อนที่พอร์ตซ้ำกันเลย (แก้ปัญหา Cold Start)

ข้อดีของ Neo4j: Node มี Label แยกชนิดอยู่แล้ว (`:Coin`, `:Category`) จึงไม่ต้องกรอง `node_type` เองเหมือนใน NetworkX

In [ ]:
categories = {
    "BTC": "Layer1",
    "ETH": "Layer1",
    "SOL": "Layer1",
    "ADA": "Layer1",
    "BNB": "Exchange",
    "XRP": "Payment",
    "DOGE": "Meme",
    "PEPE": "Meme"
}

category_data = [
    {"coin": coin, "category": cat}
    for coin, cat in categories.items()
]

driver.execute_query(
    """
    CREATE CONSTRAINT category_name_unique
    IF NOT EXISTS

    FOR (k:Category)

    REQUIRE k.name IS UNIQUE
    """,
    database_=DATABASE
)

driver.execute_query(
    """
    UNWIND $rows AS row

    MATCH (c:Coin {symbol: row.coin})

    MERGE (k:Category {name: row.category})

    MERGE (c)-[:IN_CATEGORY]->(k)
    """,

    rows=category_data,
    database_=DATABASE
)

print("สร้าง Category และ IN_CATEGORY สำเร็จ")

In [ ]:
def recommend_by_category(target_user):
    """
    แนะนำโดยไม่ต้องพึ่งผู้ใช้คนอื่นเลย เดินแค่

        target_user -> เหรียญที่ถือ -> หมวดหมู่ -> เหรียญอื่นในหมวดเดียวกัน

    ใช้แก้ปัญหา Cold Start: ผู้ใช้ใหม่ที่ยังไม่มีใครถือเหรียญซ้ำกัน
    """

    result = driver.execute_query(
        """
        MATCH
            (me:User {name: $name})
            -[:HOLDS]->
            (:Coin)
            -[:IN_CATEGORY]->
            (:Category)
            <-[:IN_CATEGORY]-
            (rec:Coin)

        WHERE NOT EXISTS {
            MATCH (me)-[:HOLDS]->(rec)
        }

        RETURN
            rec.symbol AS coin,
            count(*) AS score

        ORDER BY
            score DESC,
            coin
        """,

        name=target_user,
        database_=DATABASE
    )

    return [
        (record["coin"], record["score"])
        for record in result.records
    ]


print(recommend_by_category("Anan"))

ดู Graph ทั้งหมดใน Neo4j Aura → Query

```
MATCH p = (:User)-[:HOLDS]->(:Coin)-[:IN_CATEGORY]->(:Category)
RETURN p
UNION
MATCH p = (:User)-[:FRIEND_OF]-(:User)
RETURN p
```

In [ ]:
# ปิด Connection เมื่อใช้งานเสร็จ

driver.close()

print("Neo4j connection closed")

Recommender System แบบ Graph ไม่ได้มองเพียงว่า “เหรียญไหนดี” แต่ดูว่า “ผู้ใช้คนนี้เชื่อมโยงกับผู้ใช้และเหรียญอื่นอย่างไร”